In [0]:
%py
print('heloo')

In [0]:
%sql
SHOW TABLES IN formula1_catalog.bronze

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"


In [0]:

bronze_table

In [0]:
%sql
describe history formula1_catalog.bronze.drivers

In [0]:
# spark.read for aditonal options to read table data
#ciucuits_df=spark.read.option('versionAsOf',0).table(bronze_table)

In [0]:
drivers_df=spark.table(bronze_table)

In [0]:
drivers_df_selected=drivers_df.drop("url")

In [0]:
drivers_renamed_df=(
    drivers_df_selected
        .withColumnsRenamed ({
                     "driverId":"driver_id",
                     "dateOfBirth":"date_of_birth"
                     })  
                    
)

In [0]:
from pyspark.sql import functions as F
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter("circuit_id is not  null")
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter(circuits_renamed_df['circuit_id'].isNotNull())
drivers_renamed_nulldroped_df=drivers_renamed_df.filter(
    F.col('driver_id').isNotNull()
)

In [0]:
display(drivers_renamed_nulldroped_df.count())
display(drivers_renamed_df.count())


In [0]:
dirver_concatenated_df=(
    drivers_renamed_nulldroped_df
    .withColumn("driver_name",F.initcap(F.concat(F.col("name.givenName"),F.lit(" "),F.col("name.familyName"))))
    .drop("name")
)

In [0]:
#circuits_distinct_df=circuits_renamed_nulldroped_df.distinct()
dirver_distinct_df=dirver_concatenated_df.dropDuplicates(["driver_id"])
display(dirver_distinct_df)

In [0]:

# duplicates = races_renamed_df[["season", "round"]].groupBy("season", "round").count().filter("count > 1")
# display(duplicates)

In [0]:
from pyspark.sql.functions import initcap
dirver_final_df=(dirver_distinct_df
    .withColumn('nationality',F.initcap(F.col('nationality')))
 )

In [0]:
display(dirver_final_df)

In [0]:
(
    dirver_final_df
        .write
        .format("delta")
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1_catalog.silver.drivers